<a href="https://colab.research.google.com/github/Lefty1995/Progetti-Epicode/blob/main/Real_Estate_AI_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real Estate AI Assistant - Sintesi del progetto

Questo notebook realizza un prototipo di assistente AI per supportare un'agenzia immobiliare nella creazione di materiali marketing a partire da un annuncio pubblicato su IAD Italia.

Il flusso parte dall'inserimento dell'URL dell'annuncio, estrae dati e immagini, consente una revisione manuale delle informazioni e genera contenuti brandizzati pronti all'uso: post statico, carosello, Reel verticale 9:16, brochure PDF, caption e hashtag per i social, oltre a uno ZIP finale con tutti gli asset.

## Tecnologie utilizzate

- **Google Colab**: ambiente cloud usato per eseguire il progetto senza richiedere un computer locale potente.
- **OpenRouter e modelli LLM**: utilizzati per analizzare l'annuncio, estrarre dati strutturati e generare testi marketing, descrizioni, caption e hashtag.
- **BeautifulSoup e Requests**: usati per acquisire e analizzare i contenuti dell'annuncio.
- **Pillow, ReportLab e MoviePy**: usati per creare immagini brandizzate, brochure PDF e Reel.
- **ipywidgets**: usato per permettere all'utente di selezionare immagini, modificare testi e confermare gli output prima della generazione.
- **CLIP di Hugging Face**: usato per classificare le immagini degli immobili in categorie come cucina, bagno, soggiorno, camera ed esterno.

## Perché il modello di classificazione immagini non è stato addestrato

Il modello CLIP non è stato addestrato nuovamente perché è stato usato in modalità **zero-shot**: è un modello già pre-addestrato su un grande numero di immagini e testi, capace di confrontare una foto con descrizioni come “foto di una cucina” o “foto di un bagno”.

Per addestrare un classificatore personalizzato sarebbe stato necessario disporre di un dataset ampio di immagini immobiliari già etichettate correttamente, oltre a tempo e risorse computazionali maggiori. Per un prototipo Colab, l'approccio zero-shot è più rapido, flessibile e adatto a categorie variabili.

Il progetto mantiene comunque una fase di correzione manuale delle categorie: questo permette il controllo umano dei risultati e costituisce una possibile base per creare in futuro un dataset proprietario e addestrare un modello specifico per il settore immobiliare.

**“Per rigenerare i contenuti AI, configurare un Secret Colab chiamato Real_Estate_AI_Assistant con una chiave personale”**

# Step 1 - Setup Base
Installiamo le librerie e creiamo le cartelle locali del progetto.


In [ ]:
!pip install -q pandas numpy pillow opencv-python matplotlib scikit-learn reportlab moviepy
!pip install -q transformers torch torchvision sentencepiece beautifulsoup4 requests lxml openai webcolors

# STEP 1.1 - CARTELLE PROGETTO LOCALI


In [ ]:
import os

BASE_DIR = "/content/real_estate_ai_assistant"

folders = [
    "brand",
    "input",
    "input/images",
    "input/listings",
    "output",
    "output/brochures",
    "output/posts",
    "output/carousels",
    "output/reels",
    "temp"
]

os.makedirs(BASE_DIR, exist_ok=True)

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Progetto creato in:", BASE_DIR)
for folder in folders:
    print("-", os.path.join(BASE_DIR, folder))

# STEP 2 - CONFIGURAZIONE OPENROUTER


In [ ]:

from google.colab import userdata
from openai import OpenAI

# Recupera la chiave dai Secrets di Google Colab
OPENROUTER_API_KEY = userdata.get("Real_Estate_AI_Assistant")

if not OPENROUTER_API_KEY:
    raise ValueError(
        "Chiave non trovata. Verifica che il Secret si chiami: "
        "Real_Estate_AI_Assistant"
    )

# Client compatibile con l'API OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

# Modelli testuali
MODEL_PRIMARY = "google/gemma-4-31b-it:free"

MODEL_FALLBACKS = [
    "nvidia/nemotron-3-super-120b-a12b:free",
    "qwen/qwen3-next-80b-a3b-instruct:free"
]

# Portali supportati dal progetto
SUPPORTED_PORTALS = [
      "iad-italia.it"
]

print("Configurazione OpenRouter completata.")
print("Modello principale:", MODEL_PRIMARY)
print("Modelli fallback:", len(MODEL_FALLBACKS))
print("Portali supportati:", ", ".join(SUPPORTED_PORTALS))

# Step 3 - Funzione AI

In [ ]:
# Funzione AI con sistema di fallback

def ask_ai(user_prompt, system_prompt=None, temperature=0.4, max_tokens=1200):
    """
    Invia una richiesta a OpenRouter.
    Se il modello principale non risponde, prova i modelli fallback.
    """

    models_to_try = [MODEL_PRIMARY] + MODEL_FALLBACKS

    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    errors = []

    for model in models_to_try:
        try:
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens
            )

            content = response.choices[0].message.content

            return {
                "text": content,
                "model": model,
                "success": True,
                "errors": errors
            }

        except Exception as error:
            errors.append({
                "model": model,
                "error": str(error)
            })

    return {
        "text": None,
        "model": None,
        "success": False,
        "errors": errors
    }

print("Funzione AI configurata correttamente.")

# STEP 4 - PROFILO AGENZIA


In [ ]:
import json
import shutil
from google.colab import files
from PIL import Image

# Inserimento nome agenzia
agency_name = input("Inserisci il nome dell'agenzia: ").strip()

if not agency_name:
    raise ValueError("Il nome dell'agenzia non puo essere vuoto.")

# Caricamento logo
uploaded_logo = files.upload()

if not uploaded_logo:
    raise ValueError("Nessun logo caricato.")

uploaded_filename = next(iter(uploaded_logo))
logo_path = os.path.join(BASE_DIR, "brand", "agency_logo.png")

shutil.copy(
    uploaded_filename,
    logo_path
)

# Estrazione automatica dei colori principali
logo_image = Image.open(logo_path).convert("RGB")
logo_image.thumbnail((500, 500))

quantized_image = logo_image.quantize(colors=5)
palette = quantized_image.getpalette()
color_counts = quantized_image.getcolors()

color_counts = sorted(
    color_counts,
    key=lambda item: item[0],
    reverse=True
)

brand_colors = []

for count, color_index in color_counts:
    red = palette[color_index * 3]
    green = palette[color_index * 3 + 1]
    blue = palette[color_index * 3 + 2]

    hex_color = "#{:02X}{:02X}{:02X}".format(
        red,
        green,
        blue
    )

    brand_colors.append(hex_color)

# Salvataggio del profilo agenzia
agency_profile = {
    "agency_name": agency_name,
    "logo_path": logo_path,
    "brand_colors": brand_colors
}

profile_path = os.path.join(
    BASE_DIR,
    "brand",
    "agency_profile.json"
)

with open(profile_path, "w", encoding="utf-8") as file:
    json.dump(
        agency_profile,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Profilo agenzia salvato correttamente.")
print("Nome:", agency_name)
print("Logo:", logo_path)
print("Colori estratti:", brand_colors)

# STEP 5 - PERSONALIZZAZIONE COLORI


In [ ]:
import json
import re

with open(profile_path, "r", encoding="utf-8") as file:
    agency_profile = json.load(file)

automatic_colors = agency_profile["brand_colors"]

# Rimuove duplicati mantenendo l'ordine originale
clean_colors = []

for color in automatic_colors:
    color = color.upper()

    if color not in clean_colors:
        clean_colors.append(color)

# Rimuove il bianco quasi puro, spesso dovuto allo sfondo del logo
filtered_colors = []

for color in clean_colors:
    red = int(color[1:3], 16)
    green = int(color[3:5], 16)
    blue = int(color[5:7], 16)

    if red + green + blue < 735:
        filtered_colors.append(color)

if filtered_colors:
    clean_colors = filtered_colors

print("Colori estratti automaticamente:")
print(", ".join(clean_colors))

manual_colors = input(
    "Inserisci colori personalizzati HEX separati da virgola "
    "oppure premi INVIO per mantenere quelli automatici: "
).strip()

if manual_colors:
    selected_colors = []

    for color in manual_colors.split(","):
        color = color.strip().upper()

        if re.fullmatch(r"#[0-9A-F]{6}", color):
            if color not in selected_colors:
                selected_colors.append(color)

    if not selected_colors:
        raise ValueError(
            "Nessun colore valido. Usa il formato #RRGGBB."
        )
else:
    selected_colors = clean_colors

agency_profile["brand_colors"] = selected_colors

with open(profile_path, "w", encoding="utf-8") as file:
    json.dump(
        agency_profile,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Colori salvati correttamente:")
print(", ".join(selected_colors))

# STEP 6 - INSERIMENTO ANNUNCIO


In [ ]:
from urllib.parse import urlparse

listing_url = input(
    "Incolla il link dell'annuncio immobiliare: "
).strip()

if not listing_url.startswith(("http://", "https://")):
    raise ValueError(
        "Inserisci un URL valido che inizi con http:// o https://"
    )

parsed_url = urlparse(listing_url)
domain = parsed_url.netloc.lower().replace("www.", "")

portal = None

for supported_portal in SUPPORTED_PORTALS:
    if supported_portal in domain:
        portal = supported_portal
        break

if portal is None:
    raise ValueError(
        "Portale non supportato. Usa Immobiliare.it, Idealista o IAD Italia."
    )

listing_data = {
    "url": listing_url,
    "portal": portal,
    "domain": domain
}

listing_path = os.path.join(
    BASE_DIR,
    "input",
    "listings",
    "current_listing.json"
)

with open(listing_path, "w", encoding="utf-8") as file:
    json.dump(
        listing_data,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Annuncio acquisito correttamente.")
print("Portale riconosciuto:", portal)
print("URL salvato in:", listing_path)

# Step 7 - Acquisizione Annuncio

In [ ]:
import json
import re
import html as html_lib
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

with open(listing_path, "r", encoding="utf-8") as file:
    listing_data = json.load(file)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/120 Safari/537.36"
    )
}

response = requests.get(
    listing_data["url"],
    headers=headers,
    timeout=30
)

response.raise_for_status()

html_content = response.text
soup = BeautifulSoup(html_content, "html.parser")

page_title = soup.title.get_text(strip=True) if soup.title else ""

meta_description = soup.find(
    "meta",
    attrs={"name": "description"}
)

description = (
    meta_description.get("content", "").strip()
    if meta_description
    else ""
)

image_urls = []
seen_image_paths = set()

def add_image_url(candidate):
    if not candidate:
        return

    candidate = html_lib.unescape(candidate)

    # Decodifica gli URL presenti nel payload Nuxt
    candidate = candidate.replace("\\u002F", "/")
    candidate = candidate.replace("\\/", "/")
    candidate = candidate.strip().strip("\"'")

    if candidate.startswith("//"):
        candidate = "https:" + candidate

    full_url = urljoin(listing_data["url"], candidate)
    parsed_url = urlparse(full_url)

    valid_host = parsed_url.netloc.lower() in [
        "images.iad-italia.it",
        "images.playiad.com"
    ]

    valid_path = parsed_url.path.startswith(
        "/real_estate_pictures/"
    )

    if not valid_host or not valid_path:
        return

    # Usa il percorso per eliminare le varianti della stessa foto
    image_key = parsed_url.path.lower()

    if image_key in seen_image_paths:
        return

    seen_image_paths.add(image_key)

    # Mantiene l'URL senza parametri di dimensione o formato
    clean_url = (
        f"{parsed_url.scheme}://"
        f"{parsed_url.netloc}"
        f"{parsed_url.path}"
    )

    image_urls.append(clean_url)

# Cerca immagini nei tag HTML
image_attributes = [
    "src",
    "data-src",
    "data-original",
    "data-lazy-src",
    "data-image",
    "data-full",
    "data-fsrc",
    "data-photo"
]

for image_tag in soup.find_all("img"):
    for attribute in image_attributes:
        add_image_url(image_tag.get(attribute))

    srcset = image_tag.get("srcset")

    if srcset:
        for item in srcset.split(","):
            srcset_url = item.strip().split(" ")[0]
            add_image_url(srcset_url)

# Cerca immagini nei meta tag
for meta in soup.find_all("meta"):
    property_name = (
        meta.get("property", "")
        or meta.get("name", "")
    ).lower()

    if "image" in property_name:
        add_image_url(meta.get("content"))

# Cerca le foto dentro il payload Nuxt
image_pattern = re.compile(
    r"https?://"
    r"(?:images\.iad-italia\.it|images\.playiad\.com)"
    r"/real_estate_pictures/"
    r"[^\"'<>\\\s]+",
    re.IGNORECASE
)

for script in soup.find_all("script"):
    script_text = script.get_text()

    script_text = script_text.replace(
        "\\u002F",
        "/"
    )

    script_text = script_text.replace(
        "\\/",
        "/"
    )

    for match in image_pattern.findall(script_text):
        add_image_url(match)

page_text = soup.get_text(
    separator=" ",
    strip=True
)

raw_listing = {
    "url": listing_data["url"],
    "portal": listing_data["portal"],
    "title": page_title,
    "description": description,
    "page_text": page_text[:20000],
    "image_urls": image_urls,
    "image_count": len(image_urls)
}

raw_listing_path = os.path.join(
    BASE_DIR,
    "input",
    "listings",
    "raw_listing.json"
)

with open(raw_listing_path, "w", encoding="utf-8") as file:
    json.dump(
        raw_listing,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Pagina acquisita correttamente.")
print("Titolo:", page_title[:150])
print("Immagini uniche trovate:", len(image_urls))
print("Dati salvati in:", raw_listing_path)

# Step 7.1 - Correzione URL Immagini

In [ ]:
import json
import os
from urllib.parse import (
    urlparse,
    urlunparse,
    parse_qsl,
    urlencode
)

# Step 7.1 can run independently after Step 7 has saved the listing file.
if "raw_listing_path" not in globals():
    raw_listing_path = os.path.join(
        BASE_DIR,
        "input",
        "listings",
        "raw_listing.json"
    )

if not os.path.exists(raw_listing_path):
    raise FileNotFoundError(
        "Non trovo raw_listing.json. Esegui prima lo Step 7 - Acquisizione Annuncio."
    )

with open(raw_listing_path, "r", encoding="utf-8") as file:
    raw_listing = json.load(file)

fixed_image_urls = []

for image_url in raw_listing["image_urls"]:
    parsed_url = urlparse(image_url)

    if parsed_url.netloc.lower() == "images.iad-italia.it":
        query_params = dict(parse_qsl(parsed_url.query))

        query_params.setdefault("format", "auto")
        query_params.setdefault("width", "1560")

        parsed_url = parsed_url._replace(
            query=urlencode(query_params)
        )

        image_url = urlunparse(parsed_url)

    fixed_image_urls.append(image_url)

raw_listing["image_urls"] = fixed_image_urls
raw_listing["image_count"] = len(fixed_image_urls)

with open(raw_listing_path, "w", encoding="utf-8") as file:
    json.dump(
        raw_listing,
        file,
        indent=4,
        ensure_ascii=False
    )

print("URL delle immagini corretti.")
print("Totale immagini:", len(fixed_image_urls))

print("\nPrime tre URL corrette:")
for image_url in fixed_image_urls[:3]:
    print(image_url)

# 7.2 Visualizzazione Immagini

In [ ]:
import json
import requests
import matplotlib.pyplot as plt

from io import BytesIO
from PIL import Image

with open(raw_listing_path, "r", encoding="utf-8") as file:
    raw_listing = json.load(file)

image_urls = raw_listing["image_urls"]

columns = 4
rows = (len(image_urls) + columns - 1) // columns

fig, axes = plt.subplots(
    rows,
    columns,
    figsize=(16, rows * 4)
)

axes = axes.flatten()

for index, image_url in enumerate(image_urls):
    try:
        image_response = requests.get(
            image_url,
            headers=headers,
            timeout=20
        )

        image = Image.open(
            BytesIO(image_response.content)
        ).convert("RGB")

        axes[index].imshow(image)
        axes[index].set_title(
            f"Immagine {index + 1}",
            fontsize=10
        )

    except Exception:
        axes[index].text(
            0.5,
            0.5,
            "Immagine non caricata",
            ha="center",
            va="center"
        )

    axes[index].axis("off")

for index in range(len(image_urls), len(axes)):
    axes[index].axis("off")

plt.tight_layout()
plt.show()

print("URL delle immagini trovate:\n")

for index, image_url in enumerate(image_urls, start=1):
    print(f"{index}. {image_url}")

# Step 8 - Selezione Immagini

In [ ]:
import os
import json
import requests
import ipywidgets as widgets
import matplotlib.pyplot as plt

from io import BytesIO
from PIL import Image
from IPython.display import display
from google.colab import output

output.enable_custom_widget_manager()

with open(raw_listing_path, "r", encoding="utf-8") as file:
    raw_listing = json.load(file)

image_urls = raw_listing["image_urls"]

selected_dir = os.path.join(
    BASE_DIR,
    "input",
    "images",
    "selected"
)

os.makedirs(selected_dir, exist_ok=True)

def download_preview(image_url):
    try:
        response = requests.get(
            image_url,
            headers=headers,
            timeout=20
        )

        response.raise_for_status()

        image = Image.open(
            BytesIO(response.content)
        ).convert("RGB")

        image.thumbnail((220, 160))

        preview_buffer = BytesIO()
        image.save(preview_buffer, format="JPEG")

        return preview_buffer.getvalue()

    except Exception:
        return None

image_checkboxes = []
image_rows = []

for index, image_url in enumerate(image_urls, start=1):
    preview_bytes = download_preview(image_url)

    if preview_bytes:
        image_widget = widgets.Image(
            value=preview_bytes,
            format="jpeg",
            width=220,
            height=160
        )
    else:
        image_widget = widgets.Label(
            value="Anteprima non disponibile",
            layout=widgets.Layout(
                width="220px",
                height="160px"
            )
        )

    checkbox = widgets.Checkbox(
        value=True,
        description=f"Foto {index}",
        indent=False,
        layout=widgets.Layout(width="100px")
    )

    image_checkboxes.append({
        "index": index,
        "url": image_url,
        "checkbox": checkbox
    })

    image_rows.append(
        widgets.VBox([
            image_widget,
            checkbox
        ])
    )

gallery_grid = widgets.GridBox(
    image_rows,
    layout=widgets.Layout(
        grid_template_columns="repeat(4, 250px)",
        grid_gap="20px",
        padding="10px"
    )
)

manual_upload = widgets.FileUpload(
    accept="image/*",
    multiple=True,
    description="Aggiungi immagini"
)

upload_status = widgets.Label(
    value="Nessuna immagine aggiuntiva caricata."
)

def update_upload_status(change):
    uploaded_files = manual_upload.value

    if isinstance(uploaded_files, dict):
        count = len(uploaded_files)
    else:
        count = len(uploaded_files)

    upload_status.value = (
        f"Immagini aggiuntive selezionate: {count}"
    )

manual_upload.observe(
    update_upload_status,
    names="value"
)

confirm_button = widgets.Button(
    description="Conferma selezione",
    button_style="success",
    icon="check"
)

result_output = widgets.Output()

def get_uploaded_files():
    uploaded_files = manual_upload.value

    if isinstance(uploaded_files, dict):
        return [
            (name, data["content"])
            for name, data in uploaded_files.items()
        ]

    return [
        (data["name"], data["content"])
        for data in uploaded_files
    ]

def save_selected_images(button):
    selected_manifest = []
    saved_count = 0

    with result_output:
        result_output.clear_output()

        # Salvataggio delle immagini estratte e selezionate
        for item in image_checkboxes:
            if not item["checkbox"].value:
                continue

            try:
                response = requests.get(
                    item["url"],
                    headers=headers,
                    timeout=30
                )

                response.raise_for_status()

                image = Image.open(
                    BytesIO(response.content)
                ).convert("RGB")

                file_name = f"image_{item['index']:03d}.jpg"
                file_path = os.path.join(
                    selected_dir,
                    file_name
                )

                image.save(file_path, "JPEG", quality=95)

                selected_manifest.append({
                    "file_name": file_name,
                    "file_path": file_path,
                    "source": "listing_url",
                    "original_url": item["url"]
                })

                saved_count += 1

            except Exception as error:
                print(
                    f"Errore immagine {item['index']}: {error}"
                )

        # Salvataggio delle immagini caricate manualmente
        for manual_index, (file_name, file_content) in enumerate(
            get_uploaded_files(),
            start=1
        ):
            try:
                image = Image.open(
                    BytesIO(file_content)
                ).convert("RGB")

                saved_name = (
                    f"manual_{manual_index:03d}.jpg"
                )

                file_path = os.path.join(
                    selected_dir,
                    saved_name
                )

                image.save(file_path, "JPEG", quality=95)

                selected_manifest.append({
                    "file_name": saved_name,
                    "file_path": file_path,
                    "source": "manual_upload",
                    "original_name": file_name
                })

                saved_count += 1

            except Exception as error:
                print(
                    f"Errore immagine manuale {file_name}: {error}"
                )

        selected_images_path = os.path.join(
            BASE_DIR,
            "input",
            "listings",
            "selected_images.json"
        )

        with open(
            selected_images_path,
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(
                selected_manifest,
                file,
                indent=4,
                ensure_ascii=False
            )

        print(
            f"Selezione completata: {saved_count} immagini salvate."
        )

        print(
            "Manifest salvato in:",
            selected_images_path
        )

confirm_button.on_click(save_selected_images)

display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Seleziona le immagini da utilizzare</h3>"
        ),
        gallery_grid,
        widgets.HTML(
            "<h4>Aggiungi immagini dal computer</h4>"
        ),
        manual_upload,
        upload_status,
        confirm_button,
        result_output
    ])
)

# Step 9 - Classificazione Immagini

In [ ]:
!pip install -q -U transformers huggingface_hub safetensors

In [ ]:
import os

os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

import json
import time
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

# Aumenta i timeout prima di importare/usare Hugging Face
os.environ["HF_HUB_ETAG_TIMEOUT"] = "60"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

MODEL_ID = "openai/clip-vit-base-patch32"
CACHE_DIR = "/content/huggingface_cache"

device = "cuda" if torch.cuda.is_available() else "cpu"

for attempt in range(3):
    try:
        print(f"Caricamento modello, tentativo {attempt + 1}/3...")

        clip_model = CLIPModel.from_pretrained(
            MODEL_ID,
            cache_dir=CACHE_DIR
        ).to(device)

        clip_processor = CLIPProcessor.from_pretrained(
            MODEL_ID,
            cache_dir=CACHE_DIR
        )

        print("Modello caricato correttamente.")
        break

    except Exception as error:
        print(f"Errore di caricamento: {error}")

        if attempt == 2:
            raise RuntimeError(
                "Impossibile scaricare il modello da Hugging Face. "
                "Riprovare più tardi o riavviare il runtime Colab."
            ) from error

        time.sleep(10 * (attempt + 1))
with open(
    os.path.join(
        BASE_DIR,
        "input",
        "listings",
        "selected_images.json"
    ),
    "r",
    encoding="utf-8"
) as file:
    selected_images = json.load(file)

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

categories = {
    "cucina": "a real estate photo of a kitchen",
    "bagno": "a real estate photo of a bathroom",
    "soggiorno": "a real estate photo of a living room",
    "camera": "a real estate photo of a bedroom",
    "facciata": "a photo of the facade of a house",
    "esterno": "a photo of a garden or outdoor area",
    "garage": "a photo of a garage",
    "taverna": "a photo of a basement or tavern",
    "terreno": "a photo of an empty land plot",
    "sala da pranzo": "a photo of a dining room",
    "ufficio": "a photo of a home office",
    "corridoio": "a photo of a hallway",
    "balcone": "a photo of a balcony or terrace"
}

category_names = list(categories.keys())
category_prompts = list(categories.values())

print("Categorie aggiornate:")
for category in category_names:
    print("-", category)

classified_images = []

for item in selected_images:
    image_path = item["file_path"]

    try:
        image = Image.open(image_path).convert("RGB")

        inputs = clip_processor(
            text=category_prompts,
            images=image,
            return_tensors="pt",
            padding=True
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = clip_model(**inputs)
            probabilities = outputs.logits_per_image.softmax(
                dim=1
            )[0]

        best_index = probabilities.argmax().item()
        best_category = category_names[best_index]
        confidence = float(probabilities[best_index].item())

        scores = {
            category_names[index]: round(
                float(probabilities[index].item()),
                4
            )
            for index in range(len(category_names))
        }

        classified_images.append({
            **item,
            "category": best_category,
            "confidence": round(confidence, 4),
            "scores": scores
        })

        print(
            item["file_name"],
            "->",
            best_category,
            f"({confidence:.2%})"
        )

    except Exception as error:
        print(
            "Errore nella classificazione di",
            item["file_name"],
            ":",
            error
        )

classified_images_path = os.path.join(
    BASE_DIR,
    "input",
    "listings",
    "classified_images.json"
)

with open(
    classified_images_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        classified_images,
        file,
        indent=4,
        ensure_ascii=False
    )

print("\nClassificazione completata.")
print("Immagini classificate:", len(classified_images))
print("Risultati salvati in:", classified_images_path)
print("Dispositivo utilizzato:", device)

# Step 10 - Correzione Classificazioni

In [ ]:
import os
import json
import ipywidgets as widgets

from PIL import Image
from io import BytesIO
from IPython.display import display

classified_images_path = os.path.join(
    BASE_DIR,
    "input",
    "listings",
    "classified_images.json"
)

with open(
    classified_images_path,
    "r",
    encoding="utf-8"
) as file:
    classified_images = json.load(file)

category_options = list(categories.keys())
classification_widgets = []
image_cards = []

for item in classified_images:
    image_path = item["file_path"]

    try:
        image = Image.open(image_path).convert("RGB")
        image.thumbnail((240, 170))

        preview_buffer = BytesIO()
        image.save(preview_buffer, format="JPEG")

        image_widget = widgets.Image(
            value=preview_buffer.getvalue(),
            format="jpeg",
            width=240,
            height=170
        )

    except Exception:
        image_widget = widgets.Label(
            value="Immagine non disponibile",
            layout=widgets.Layout(
                width="240px",
                height="170px"
            )
        )

    current_category = item["category"]

    if current_category not in category_options:
        current_category = category_options[0]

    category_dropdown = widgets.Dropdown(
        options=category_options,
        value=current_category,
        description="AI:",
        layout=widgets.Layout(width="250px")
    )

    custom_category = widgets.Text(
        value="",
        placeholder="es. lavanderia, ripostiglio...",
        description="Manuale:",
        layout=widgets.Layout(width="250px")
    )

    confidence_label = widgets.Label(
        value=(
            f"Confidenza AI: "
            f"{item.get('confidence', 0):.2%}"
        )
    )

    classification_widgets.append({
        "item": item,
        "dropdown": category_dropdown,
        "custom_category": custom_category
    })

    image_cards.append(
        widgets.VBox([
            image_widget,
            widgets.Label(value=item["file_name"]),
            confidence_label,
            category_dropdown,
            custom_category
        ])
    )

classification_grid = widgets.GridBox(
    image_cards,
    layout=widgets.Layout(
        grid_template_columns="repeat(3, 290px)",
        grid_gap="25px",
        padding="10px"
    )
)

save_button = widgets.Button(
    description="Salva classificazioni",
    button_style="success",
    icon="save"
)

result_output = widgets.Output()

def save_corrected_classifications(button):
    corrected_images = []

    for element in classification_widgets:
        item = element["item"]

        selected_category = (
            element["custom_category"].value.strip().lower()
        )

        if not selected_category:
            selected_category = element["dropdown"].value

        corrected_item = {
            **item,
            "original_category": item["category"],
            "corrected_category": selected_category,
            "category": selected_category,
            "manually_corrected": (
                selected_category != item["category"]
            )
        }

        corrected_images.append(corrected_item)

    corrected_images_path = os.path.join(
        BASE_DIR,
        "input",
        "listings",
        "corrected_images.json"
    )

    with open(
        corrected_images_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            corrected_images,
            file,
            indent=4,
            ensure_ascii=False
        )

    with result_output:
        result_output.clear_output()
        print("Classificazioni salvate correttamente.")
        print("File:", corrected_images_path)

save_button.on_click(save_corrected_classifications)

display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Controlla e correggi le categorie</h3>"
        ),
        classification_grid,
        save_button,
        result_output
    ])
)

# Step 11 - Analisi Dati Annuncio

In [ ]:
import os
import json
import re

with open(
    raw_listing_path,
    "r",
    encoding="utf-8"
) as file:
    raw_listing = json.load(file)

system_prompt = """
Sei un assistente specializzato in annunci immobiliari italiani.

Analizza il testo fornito e restituisci esclusivamente un oggetto JSON
valido, senza spiegazioni e senza blocchi markdown.

Non inventare informazioni mancanti.
Se un dato non è disponibile, usa null.
I punti di forza devono essere restituiti come lista.
"""

user_prompt = f"""
Analizza questo annuncio immobiliare.

Titolo:
{raw_listing.get("title", "")}

Descrizione:
{raw_listing.get("description", "")}

Testo completo:
{raw_listing.get("page_text", "")[:16000]}

Restituisci questo formato JSON:

{{
    "titolo": null,
    "tipologia": null,
    "localita": null,
    "prezzo": null,
    "superficie_mq": null,
    "locali": null,
    "camere": null,
    "bagni": null,
    "terreno_mq": null,
    "anno_costruzione": null,
    "punti_forza": [],
    "descrizione_breve": null,
    "target_acquirente": null
}}
"""

ai_response = ask_ai(
    user_prompt=user_prompt,
    system_prompt=system_prompt,
    temperature=0.2,
    max_tokens=1800
)

if not ai_response["success"]:
    raise RuntimeError(
        f"Errore nella risposta AI: {ai_response['errors']}"
    )

response_text = ai_response["text"].strip()

# Rimuove eventuali blocchi markdown restituiti dal modello
response_text = re.sub(
    r"```json|```",
    "",
    response_text,
    flags=re.IGNORECASE
).strip()

# Estrae il primo oggetto JSON presente nella risposta
json_start = response_text.find("{")
json_end = response_text.rfind("}")

if json_start == -1 or json_end == -1:
    raise ValueError(
        "La risposta AI non contiene un oggetto JSON valido."
    )

json_text = response_text[json_start:json_end + 1]
structured_listing = json.loads(json_text)

structured_listing_path = os.path.join(
    BASE_DIR,
    "input",
    "listings",
    "structured_listing.json"
)

with open(
    structured_listing_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        structured_listing,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Analisi dell'annuncio completata.")
print("Modello utilizzato:", ai_response["model"])
print("Dati strutturati salvati in:", structured_listing_path)
print(json.dumps(
    structured_listing,
    indent=4,
    ensure_ascii=False
))

# Step 11.a - Analisi Visiva Immagini



In [ ]:
import os
import json
import torch

from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

corrected_images_path = os.path.join(
    BASE_DIR,
    "input",
    "listings",
    "corrected_images.json"
)

with open(
    corrected_images_path,
    "r",
    encoding="utf-8"
) as file:
    corrected_images = json.load(file)

device = "cuda" if torch.cuda.is_available() else "cpu"

visual_processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

visual_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

visual_descriptions = []

for item in corrected_images:
    try:
        image = Image.open(
            item["file_path"]
        ).convert("RGB")

        prompt = (
            "a real estate photo of a "
            f"{item['category']}"
        )

        inputs = visual_processor(
            images=image,
            text=prompt,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            output_ids = visual_model.generate(
                **inputs,
                max_new_tokens=40,
                num_beams=4
            )

        caption = visual_processor.decode(
            output_ids[0],
            skip_special_tokens=True
        )

        visual_descriptions.append({
            "file_name": item["file_name"],
            "file_path": item["file_path"],
            "category": item["category"],
            "visual_description": caption
        })

        print(
            item["file_name"],
            "->",
            caption
        )

    except Exception as error:
        print(
            "Errore con",
            item["file_name"],
            ":",
            error
        )

visual_descriptions_path = os.path.join(
    BASE_DIR,
    "input",
    "listings",
    "visual_descriptions.json"
)

with open(
    visual_descriptions_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        visual_descriptions,
        file,
        indent=4,
        ensure_ascii=False
    )

print("\nAnalisi visiva completata.")
print(
    "Descrizioni salvate in:",
    visual_descriptions_path
)
print("Dispositivo utilizzato:", device)

# Step 12 - Contenuti Social

In [ ]:
import os
import json
import re

with open(
    structured_listing_path,
    "r",
    encoding="utf-8"
) as file:
    structured_listing = json.load(file)

corrected_images_path = os.path.join(
    BASE_DIR,
    "input",
    "listings",
    "corrected_images.json"
)

with open(
    corrected_images_path,
    "r",
    encoding="utf-8"
) as file:
    corrected_images = json.load(file)

image_categories = [
    {
        "file_name": item["file_name"],
        "category": item["category"]
    }
    for item in corrected_images
]

system_prompt = """
Sei un copywriter specializzato nel settore immobiliare italiano.

Crea testi professionali, chiari e persuasivi.
Non inventare dati non presenti nell'annuncio.
Usa un tono diverso per ogni piattaforma.
Restituisci esclusivamente JSON valido.
"""

user_prompt = f"""
Nome agenzia:
{agency_profile["agency_name"]}

Dati dell'immobile:
{json.dumps(
    structured_listing,
    ensure_ascii=False,
    indent=2
)}

Categorie delle immagini:
{json.dumps(
    image_categories,
    ensure_ascii=False,
    indent=2
)}

Genera questo formato JSON:

{{
    "facebook_post": "",
    "instagram_caption": "",
    "linkedin_post": "",
    "simple_post_text": "",
    "carousel_title": "",
    "carousel_slides": [
        {{
            "category": "",
            "overlay_text": ""
        }}
    ]
}}

Regole:

- Facebook: testo coinvolgente e informativo.
- Instagram: testo breve, emozionale e con hashtag pertinenti.
- LinkedIn: tono professionale e orientato al valore dell'immobile.
- Simple post: testo molto breve.
- Carousel title: titolo commerciale breve.
- Carousel slides: crea una slide per ogni categoria disponibile.
- overlay_text deve contenere la categoria e una frase breve.
- Non aggiungere caratteristiche non presenti nei dati.
"""

ai_response = ask_ai(
    user_prompt=user_prompt,
    system_prompt=system_prompt,
    temperature=0.5,
    max_tokens=2500
)

if not ai_response["success"]:
    raise RuntimeError(
        f"Errore nella generazione dei contenuti: "
        f"{ai_response['errors']}"
    )

response_text = ai_response["text"].strip()

response_text = re.sub(
    r"```json|```",
    "",
    response_text,
    flags=re.IGNORECASE
).strip()

json_start = response_text.find("{")
json_end = response_text.rfind("}")

if json_start == -1 or json_end == -1:
    raise ValueError(
        "La risposta AI non contiene JSON valido."
    )

generated_content = json.loads(
    response_text[json_start:json_end + 1]
)

generated_content_path = os.path.join(
    BASE_DIR,
    "output",
    "posts",
    "generated_content.json"
)

with open(
    generated_content_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        generated_content,
        file,
        indent=4,
        ensure_ascii=False
    )

print("Contenuti generati correttamente.")
print("Modello utilizzato:", ai_response["model"])
print("File salvato in:", generated_content_path)

# Step 13 - Scelta Contenuti

In [ ]:
import os
import json
import ipywidgets as widgets
from IPython.display import display

content_options = {
    "brochure": "Brochure PDF",
    "facebook": "Post Facebook",
    "instagram": "Post Instagram",
    "linkedin": "Post LinkedIn",
    "simple_post": "Post semplice",
    "carousel": "Carosello",
    "reel": "AI Reel Composer"
}

option_widgets = {}

for key, label in content_options.items():
    option_widgets[key] = widgets.Checkbox(
        value=True,
        description=label,
        indent=False
    )

save_options_button = widgets.Button(
    description="Conferma contenuti",
    button_style="success",
    icon="check"
)

options_output = widgets.Output()

def save_content_options(button):
    selected_options = {
        key: widget.value
        for key, widget in option_widgets.items()
    }

    selected_outputs_path = os.path.join(
        BASE_DIR,
        "input",
        "listings",
        "generation_options.json"
    )

    with open(
        selected_outputs_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            selected_options,
            file,
            indent=4,
            ensure_ascii=False
        )

    with options_output:
        options_output.clear_output()
        print("Scelta dei contenuti salvata correttamente.")
        print("File salvato in:", selected_outputs_path)

save_options_button.on_click(save_content_options)

display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Seleziona i contenuti da generare</h3>"
        ),
        *option_widgets.values(),
        save_options_button,
        options_output
    ])
)

# Step 14 - Selezione Immagini Principali

In [ ]:
import os
import json
import ipywidgets as widgets

from PIL import Image
from io import BytesIO
from IPython.display import display

with open(
    corrected_images_path,
    "r",
    encoding="utf-8"
) as file:
    corrected_images = json.load(file)

if not corrected_images:
    raise ValueError(
        "Non ci sono immagini disponibili."
    )

images_by_category = {}

for item in corrected_images:
    category = item["category"]

    if category not in images_by_category:
        images_by_category[category] = []

    images_by_category[category].append(item)

image_options = [
    (
        f"{index + 1}. {item['file_name']} "
        f"- {item['category']}",
        index
    )
    for index, item in enumerate(corrected_images)
]

hero_dropdown = widgets.Dropdown(
    options=image_options,
    value=0,
    description="Immagine:",
    layout=widgets.Layout(width="450px")
)

hero_preview = widgets.Image(
    width=500,
    height=350
)

hero_info = widgets.Label()

def update_hero_preview(change=None):
    selected_index = hero_dropdown.value
    selected_item = corrected_images[selected_index]

    image = Image.open(
        selected_item["file_path"]
    ).convert("RGB")

    image.thumbnail((500, 350))

    preview_buffer = BytesIO()
    image.save(preview_buffer, format="JPEG")

    hero_preview.value = preview_buffer.getvalue()
    hero_preview.format = "jpeg"

    hero_info.value = (
        f"Categoria: {selected_item['category']} | "
        f"File: {selected_item['file_name']}"
    )

hero_dropdown.observe(
    update_hero_preview,
    names="value"
)

update_hero_preview()

save_hero_button = widgets.Button(
    description="Salva immagine principale",
    button_style="success",
    icon="save"
)

hero_output = widgets.Output()

def save_hero_image(button):
    selected_index = hero_dropdown.value
    hero_image = corrected_images[selected_index]

    content_assets = {
        "hero_image": hero_image,
        "images_by_category": images_by_category
    }

    content_assets_path = os.path.join(
        BASE_DIR,
        "input",
        "listings",
        "content_assets.json"
    )

    with open(
        content_assets_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            content_assets,
            file,
            indent=4,
            ensure_ascii=False
        )

    with hero_output:
        hero_output.clear_output()
        print("Immagine principale salvata.")
        print("File:", hero_image["file_name"])
        print("Categoria:", hero_image["category"])

save_hero_button.on_click(save_hero_image)

display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Scegli l'immagine principale</h3>"
        ),
        hero_dropdown,
        hero_preview,
        hero_info,
        save_hero_button,
        hero_output
    ])
)

# Step 15 - Creazione Post Brandizzati

In [ ]:
import os
import json
import copy
import ipywidgets as widgets

from PIL import (
    Image,
    ImageDraw,
    ImageFont,
    ImageColor
)

from matplotlib import font_manager
from IPython.display import display

with open(
    os.path.join(
        BASE_DIR,
        "input",
        "listings",
        "generation_options.json"
    ),
    "r",
    encoding="utf-8"
) as file:
    generation_options = json.load(file)

with open(
    os.path.join(
        BASE_DIR,
        "output",
        "posts",
        "generated_content.json"
    ),
    "r",
    encoding="utf-8"
) as file:
    generated_content = json.load(file)

with open(
    os.path.join(
        BASE_DIR,
        "input",
        "listings",
        "content_assets.json"
    ),
    "r",
    encoding="utf-8"
) as file:
    content_assets = json.load(file)

with open(
    structured_listing_path,
    "r",
    encoding="utf-8"
) as file:
    structured_listing = json.load(file)

hero_image = Image.open(
    content_assets["hero_image"]["file_path"]
).convert("RGB")

logo_image = Image.open(
    agency_profile["logo_path"]
).convert("RGBA")

primary_rgb = ImageColor.getrgb(
    agency_profile["brand_colors"][0]
)

secondary_rgb = ImageColor.getrgb(
    agency_profile["brand_colors"][1]
)

bold_font_path = font_manager.findfont(
    font_manager.FontProperties(
        family="DejaVu Sans",
        weight="bold"
    )
)

regular_font_path = font_manager.findfont(
    font_manager.FontProperties(
        family="DejaVu Sans",
        weight="normal"
    )
)

font_title = ImageFont.truetype(
    bold_font_path,
    64
)

font_info = ImageFont.truetype(
    bold_font_path,
    34
)

font_description = ImageFont.truetype(
    regular_font_path,
    27
)

posts_dir = os.path.join(
    BASE_DIR,
    "output",
    "posts"
)

os.makedirs(posts_dir, exist_ok=True)

def get_value(key):
    value = structured_listing.get(key)

    if value is None or str(value).strip() == "":
        return None

    return str(value).strip()

property_type = (
    get_value("tipologia")
    or "IMMOBILE IN VENDITA"
)

location = get_value("localita")
price = get_value("prezzo")
surface = get_value("superficie_mq")
rooms = get_value("locali")
bedrooms = get_value("camere")
bathrooms = get_value("bagni")
land_surface = get_value("terreno_mq")

title_default = property_type.upper()

line_1_parts = []

if location:
    line_1_parts.append(location)

if price:
    line_1_parts.append(price)

line_1_default = " | ".join(line_1_parts)

line_2_parts = []

if rooms:
    line_2_parts.append(f"{rooms} locali")

if bedrooms:
    line_2_parts.append(f"{bedrooms} camere")

if bathrooms:
    line_2_parts.append(f"{bathrooms} bagni")

line_2_default = " | ".join(line_2_parts)

line_3_parts = []

if surface:
    line_3_parts.append(f"{surface} m²")

if land_surface:
    line_3_parts.append(
        f"Terreno {land_surface} m²"
    )

line_3_default = " | ".join(line_3_parts)

phrase_default = generated_content.get(
    "simple_post",
    generated_content.get(
        "simple_post_text",
        ""
    )
)

overlay_original = {
    "title": title_default,
    "line_1": line_1_default,
    "line_2": line_2_default,
    "line_3": line_3_default,
    "phrase": phrase_default
}

title_widget = widgets.Text(
    value=title_default,
    description="Titolo:",
    layout=widgets.Layout(width="850px")
)

line_1_widget = widgets.Text(
    value=line_1_default,
    description="Info 1:",
    layout=widgets.Layout(width="850px")
)

line_2_widget = widgets.Text(
    value=line_2_default,
    description="Info 2:",
    layout=widgets.Layout(width="850px")
)

line_3_widget = widgets.Text(
    value=line_3_default,
    description="Info 3:",
    layout=widgets.Layout(width="850px")
)

phrase_widget = widgets.Textarea(
    value=phrase_default,
    description="Frase:",
    layout=widgets.Layout(
        width="850px",
        height="90px"
    )
)

confirm_button = widgets.Button(
    description="Conferma e crea post",
    button_style="success",
    icon="check"
)

result_output = widgets.Output()

def prepare_image_without_crop(image, size):
    canvas = Image.new(
        "RGB",
        size,
        (245, 245, 245)
    )

    image_copy = image.copy()

    image_copy.thumbnail(
        size,
        Image.Resampling.LANCZOS
    )

    x_position = (
        size[0] - image_copy.width
    ) // 2

    y_position = (
        size[1] - image_copy.height
    ) // 2

    canvas.paste(
        image_copy,
        (x_position, y_position)
    )

    return canvas

def wrap_text(draw, text, font, max_width):
    words = text.split()
    lines = []
    current_line = ""

    for word in words:
        test_line = (
            f"{current_line} {word}".strip()
        )

        if draw.textlength(
            test_line,
            font=font
        ) <= max_width:
            current_line = test_line
        else:
            if current_line:
                lines.append(current_line)

            current_line = word

    if current_line:
        lines.append(current_line)

    return lines[:2]

def create_branded_post(
    overlay,
    output_name
):
    canvas_width = 1080
    canvas_height = 1080

    panel_height = 400
    image_height = canvas_height - panel_height

    image_area = prepare_image_without_crop(
        hero_image,
        (canvas_width, image_height)
    )

    final_canvas = Image.new(
        "RGB",
        (canvas_width, canvas_height),
        (245, 245, 245)
    )

    final_canvas.paste(
        image_area,
        (0, 0)
    )

    draw = ImageDraw.Draw(final_canvas)

    draw.rectangle(
        [
            (0, image_height),
            (canvas_width, canvas_height)
        ],
        fill=primary_rgb
    )

    logo_width = 115
    logo_ratio = logo_image.height / logo_image.width
    logo_height = int(logo_width * logo_ratio)

    resized_logo = logo_image.resize(
        (logo_width, logo_height),
        Image.Resampling.LANCZOS
    )

    final_canvas.paste(
        resized_logo,
        (35, 30),
        resized_logo
    )

    draw.text(
        (50, image_height + 35),
        overlay["title"],
        fill="white",
        font=font_title
    )

    info_y = image_height + 145

    for line in [
        overlay["line_1"],
        overlay["line_2"],
        overlay["line_3"]
    ]:
        if line.strip():
            draw.text(
                (50, info_y),
                line,
                fill="white",
                font=font_info
            )

            info_y += 48

    phrase_lines = wrap_text(
        draw,
        overlay["phrase"],
        font_description,
        canvas_width - 100
    )

    phrase_y = info_y + 12

    for line in phrase_lines:
        draw.text(
            (50, phrase_y),
            line,
            fill="white",
            font=font_description
        )

        phrase_y += 36

    draw.rounded_rectangle(
        [
            (50, canvas_height - 28),
            (canvas_width - 50, canvas_height - 14)
        ],
        radius=6,
        fill=secondary_rgb
    )

    output_path = os.path.join(
        posts_dir,
        output_name
    )

    final_canvas.save(
        output_path,
        "JPEG",
        quality=98,
        subsampling=0
    )

    return output_path
def create_posts(button):
    overlay_edited = {
        "title": title_widget.value.strip(),
        "line_1": line_1_widget.value.strip(),
        "line_2": line_2_widget.value.strip(),
        "line_3": line_3_widget.value.strip(),
        "phrase": phrase_widget.value.strip()
    }

    static_post_name = "static_post.jpg"

    static_post_path = create_branded_post(
        overlay_edited,
        static_post_name
    )

    generated_post = {
        "image_path": static_post_path,
        "overlay_content": overlay_edited
    }

    generated_post_path = os.path.join(
        posts_dir,
        "static_post.json"
    )

    versions_path = os.path.join(
        posts_dir,
        "post_overlay_versions.json"
    )

    with open(
        generated_post_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            generated_post,
            file,
            indent=4,
            ensure_ascii=False
        )

    with open(
        versions_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            {
                "original_content": overlay_original,
                "edited_content": overlay_edited
            },
            file,
            indent=4,
            ensure_ascii=False
        )

    with result_output:
        result_output.clear_output()

        print("Immagine statica creata correttamente.")
        print("File:", static_post_path)
        print(
            "Versioni salvate in:",
            versions_path
        )

        display(
            Image.open(static_post_path)
        )


confirm_button.on_click(create_posts)

display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Modifica i testi presenti nelle immagini</h3>"
        ),
        title_widget,
        line_1_widget,
        line_2_widget,
        line_3_widget,
        phrase_widget,
        confirm_button,
        result_output
    ])
)

# Step 16 - Creazione Carosello

In [ ]:
import os
import json
import re
import ipywidgets as widgets

from io import BytesIO
from PIL import (
    Image,
    ImageDraw,
    ImageFont,
    ImageColor
)

from matplotlib import font_manager
from IPython.display import display

with open(
    corrected_images_path,
    "r",
    encoding="utf-8"
) as file:
    corrected_images = json.load(file)

with open(
    os.path.join(
        BASE_DIR,
        "output",
        "posts",
        "generated_content.json"
    ),
    "r",
    encoding="utf-8"
) as file:
    generated_content = json.load(file)

original_slides = generated_content.get(
    "carousel_slides",
    []
)

logo_image = Image.open(
    agency_profile["logo_path"]
).convert("RGBA")

primary_rgb = ImageColor.getrgb(
    agency_profile["brand_colors"][0]
)

secondary_rgb = ImageColor.getrgb(
    agency_profile["brand_colors"][1]
)

bold_font_path = font_manager.findfont(
    font_manager.FontProperties(
        family="DejaVu Sans",
        weight="bold"
    )
)

regular_font_path = font_manager.findfont(
    font_manager.FontProperties(
        family="DejaVu Sans",
        weight="normal"
    )
)

font_title = ImageFont.truetype(
    bold_font_path,
    58
)

font_description = ImageFont.truetype(
    regular_font_path,
    32
)

font_number = ImageFont.truetype(
    bold_font_path,
    26
)

carousels_dir = os.path.join(
    BASE_DIR,
    "output",
    "carousels"
)

os.makedirs(carousels_dir, exist_ok=True)

def remove_category_prefix(category, text):
    if not text:
        return ""

    return re.sub(
        rf"^\s*{re.escape(category)}"
        rf"\s*[:\-]?\s*",
        "",
        text.strip(),
        flags=re.IGNORECASE
    )

def get_default_description(category):
    for slide in original_slides:
        slide_category = slide.get(
            "category",
            ""
        ).strip().lower()

        if slide_category == category.lower():
            return remove_category_prefix(
                category,
                slide.get("overlay_text", "")
            )

    return ""

def create_preview(file_path):
    try:
        image = Image.open(
            file_path
        ).convert("RGB")

        image.thumbnail(
            (210, 150),
            Image.Resampling.LANCZOS
        )

        buffer = BytesIO()
        image.save(buffer, format="JPEG")

        return widgets.Image(
            value=buffer.getvalue(),
            format="jpeg",
            width=210,
            height=150
        )

    except Exception:
        return widgets.Label(
            value="Anteprima non disponibile",
            layout=widgets.Layout(
                width="210px",
                height="150px"
            )
        )

# Selezione immagini
image_selection_widgets = []
selection_cards = []

for index, item in enumerate(
    corrected_images,
    start=1
):
    checkbox = widgets.Checkbox(
        value=True,
        description=f"Immagine {index}",
        indent=False,
        layout=widgets.Layout(width="150px")
    )

    image_selection_widgets.append({
        "item": item,
        "checkbox": checkbox
    })

    selection_cards.append(
        widgets.VBox([
            create_preview(item["file_path"]),
            widgets.Label(
                value=item["category"]
            ),
            checkbox
        ])
    )

selection_grid = widgets.GridBox(
    selection_cards,
    layout=widgets.Layout(
        grid_template_columns="repeat(4, 240px)",
        grid_gap="20px",
        padding="10px"
    )
)

confirm_images_button = widgets.Button(
    description="Conferma immagini",
    button_style="primary",
    icon="check"
)

editor_container = widgets.VBox()

confirm_carousel_button = widgets.Button(
    description="Crea carosello",
    button_style="success",
    icon="picture-o",
    disabled=True
)

result_output = widgets.Output()

selected_items = []
slide_editors = []

def prepare_carousel_image(
    image_path,
    size=(1080, 1080)
):
    source_image = Image.open(
        image_path
    ).convert("RGB")

    canvas = Image.new(
        "RGB",
        size,
        (245, 245, 245)
    )

    source_image.thumbnail(
        size,
        Image.Resampling.LANCZOS
    )

    x_position = (
        size[0] - source_image.width
    ) // 2

    y_position = (
        size[1] - source_image.height
    ) // 2

    canvas.paste(
        source_image,
        (x_position, y_position)
    )

    return canvas.convert("RGBA")

def wrap_carousel_text(
    draw,
    text,
    font,
    max_width
):
    words = text.split()
    lines = []
    current_line = ""

    for word in words:
        test_line = (
            f"{current_line} {word}".strip()
        )

        if draw.textlength(
            test_line,
            font=font
        ) <= max_width:
            current_line = test_line
        else:
            if current_line:
                lines.append(current_line)

            current_line = word

    if current_line:
        lines.append(current_line)

    return lines[:3]

def create_carousel_slide(
    item,
    title,
    description,
    slide_number
):
    canvas = prepare_carousel_image(
        item["file_path"]
    )

    canvas_width, canvas_height = canvas.size
    overlay_height = 300

    overlay = Image.new(
        "RGBA",
        canvas.size,
        (0, 0, 0, 0)
    )

    overlay_draw = ImageDraw.Draw(overlay)

    overlay_draw.rectangle(
        [
            (0, canvas_height - overlay_height),
            (canvas_width, canvas_height)
        ],
        fill=(*primary_rgb, 230)
    )

    canvas = Image.alpha_composite(
        canvas,
        overlay
    )

    draw = ImageDraw.Draw(canvas)

    logo_width = 115
    logo_ratio = (
        logo_image.height / logo_image.width
    )

    logo_height = int(
        logo_width * logo_ratio
    )

    resized_logo = logo_image.resize(
        (logo_width, logo_height),
        Image.Resampling.LANCZOS
    )

    canvas.paste(
        resized_logo,
        (35, 35),
        resized_logo
    )

    draw.text(
        (canvas_width - 35, 45),
        f"{slide_number:02d}",
        fill="white",
        font=font_number,
        anchor="ra"
    )

    draw.text(
        (50, canvas_height - 240),
        title,
        fill="white",
        font=font_title
    )

    description_lines = wrap_carousel_text(
        draw,
        description,
        font_description,
        canvas_width - 100
    )

    text_y = canvas_height - 165

    for line in description_lines:
        draw.text(
            (50, text_y),
            line,
            fill="white",
            font=font_description
        )

        text_y += 40

    draw.rounded_rectangle(
        [
            (50, canvas_height - 28),
            (canvas_width - 50, canvas_height - 14)
        ],
        radius=6,
        fill=secondary_rgb
    )

    slide_path = os.path.join(
        carousels_dir,
        f"carousel_slide_{slide_number:02d}.jpg"
    )

    canvas.convert("RGB").save(
        slide_path,
        "JPEG",
        quality=98,
        subsampling=0
    )

    return slide_path

def prepare_text_editors(button):
    global selected_items
    global slide_editors

    selected_items = [
        item["item"]
        for item in image_selection_widgets
        if item["checkbox"].value
    ]

    if not selected_items:
        with result_output:
            result_output.clear_output()
            print(
                "Seleziona almeno un'immagine."
            )
        return

    slide_editors = []
    editor_elements = [
        widgets.HTML(
            "<h3>Modifica i testi delle slide</h3>"
        )
    ]

    for index, item in enumerate(
        selected_items,
        start=1
    ):
        category = item["category"]

        title_widget = widgets.Text(
            value=category.upper(),
            description=f"Titolo {index}:",
            layout=widgets.Layout(width="850px")
        )

        description_widget = widgets.Textarea(
            value=get_default_description(
                category
            ),
            description=f"Testo {index}:",
            layout=widgets.Layout(
                width="850px",
                height="90px"
            )
        )

        slide_editors.append({
            "item": item,
            "title": title_widget,
            "description": description_widget
        })

        editor_elements.extend([
            widgets.HTML(
                f"<b>Slide {index} - {category}</b>"
            ),
            title_widget,
            description_widget
        ])

    editor_container.children = editor_elements
    confirm_carousel_button.disabled = False

    with result_output:
        result_output.clear_output()
        print(
            f"{len(selected_items)} immagini selezionate."
        )
        print(
            "Ora puoi modificare i testi delle slide."
        )

def create_carousel(button):
    edited_slides = []
    carousel_manifest = []

    for index, editor in enumerate(
        slide_editors,
        start=1
    ):
        item = editor["item"]

        title = editor["title"].value.strip()
        description = (
            editor["description"].value.strip()
        )

        slide_path = create_carousel_slide(
            item,
            title,
            description,
            index
        )

        edited_slides.append({
            "category": item["category"],
            "title": title,
            "description": description,
            "image_source": item["file_path"]
        })

        carousel_manifest.append({
            "slide_number": index,
            "category": item["category"],
            "title": title,
            "description": description,
            "image_source": item["file_path"],
            "output_path": slide_path
        })

    versions_path = os.path.join(
        carousels_dir,
        "carousel_versions.json"
    )

    manifest_path = os.path.join(
        carousels_dir,
        "carousel_manifest.json"
    )

    with open(
        versions_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            {
                "original_content": original_slides,
                "edited_content": edited_slides
            },
            file,
            indent=4,
            ensure_ascii=False
        )

    with open(
        manifest_path,
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(
            carousel_manifest,
            file,
            indent=4,
            ensure_ascii=False
        )

    with result_output:
        result_output.clear_output()

        print(
            "Carosello creato correttamente."
        )

        print(
            "Numero di slide:",
            len(carousel_manifest)
        )

        print(
            "Manifest salvato in:",
            manifest_path
        )

        for slide in carousel_manifest:
            print(
                f"\nSlide {slide['slide_number']}: "
                f"{slide['category']}"
            )

            display(
                Image.open(
                    slide["output_path"]
                )
            )

confirm_images_button.on_click(
    prepare_text_editors
)

confirm_carousel_button.on_click(
    create_carousel
)

display(
    widgets.VBox([
        widgets.HTML(
            "<h3>Seleziona le immagini del carosello</h3>"
        ),
        selection_grid,
        confirm_images_button,
        editor_container,
        confirm_carousel_button,
        result_output
    ])
)


# Step 17 Creazione Reel



In [ ]:
!pip install -q moviepy ipywidgets imageio-ffmpeg

In [ ]:
import json
import os
from pathlib import Path

import ipywidgets as widgets
from IPython.display import Video, display
from PIL import Image, ImageDraw, ImageEnhance, ImageFilter, ImageFont, ImageOps
from matplotlib import font_manager

try:
    from moviepy.editor import ImageClip, concatenate_videoclips
except ImportError as error:
    raise ImportError(
        "Installa MoviePy prima di eseguire questo blocco: "
        "!pip install -q moviepy ipywidgets imageio-ffmpeg"
    ) from error


# Use only files from the current project and preserve active notebook data.
PROJECT_DIR = Path(globals().get("BASE_DIR", "/content/real_estate_ai_assistant"))
LISTINGS_DIR = PROJECT_DIR / "input" / "listings"
source_path = next(
    (path for path in [
        LISTINGS_DIR / "classified_images.json",
        LISTINGS_DIR / "selected_images.json",
    ] if path.exists()),
    None,
)
in_memory_images = globals().get("classified_images") or globals().get("selected_images")

if source_path:
    with source_path.open("r", encoding="utf-8") as file:
        image_items = json.load(file)
    print("File immagini usato:", source_path)
elif isinstance(in_memory_images, list) and in_memory_images:
    image_items = in_memory_images
    print("Uso le immagini gia presenti nella memoria del notebook.")
else:
    raise FileNotFoundError(
        "Non trovo i file immagini del progetto e non esiste una lista in memoria. "
        "Esegui prima lo step di selezione immagini."
    )

OUTPUT_DIR = PROJECT_DIR / "output" / "reels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not image_items:
    raise ValueError("La lista delle immagini e vuota.")


def item_path(item):
    path = Path(item.get("file_path", ""))
    return path if path.is_absolute() else PROJECT_DIR / path


valid_items = [item for item in image_items if item_path(item).exists()]
if not valid_items:
    raise FileNotFoundError("Nessuna immagine della lista e stata trovata sul disco.")

item_by_key = {}
for index, item in enumerate(valid_items):
    key = f"{index + 1:02d} - {item.get('file_name', item_path(item).name)}"
    item_by_key[key] = item


def default_slide_text(item):
    category = item.get("category", "immobile").replace("_", " ").title()
    description = (
        item.get("image_description")
        or item.get("visual_description")
        or item.get("description")
        or item.get("caption")
    )
    if description:
        return f"{category}: {str(description).strip()}"[:180]

    defaults = {
        "Cucina": "Ambiente funzionale e luminoso, ideale per vivere la casa ogni giorno.",
        "Bagno": "Spazio curato e funzionale, pensato per il massimo comfort.",
        "Soggiorno": "Un ambiente accogliente e luminoso da vivere con la famiglia.",
        "Camera": "Una camera tranquilla e confortevole per il tuo relax.",
        "Facciata": "Una facciata curata che valorizza l'identita della proprieta.",
        "Esterno": "Spazi esterni piacevoli per vivere momenti all'aria aperta.",
        "Balcone": "Uno spazio esterno ideale per godersi la luce e il panorama.",
        "Garage": "Spazio pratico e comodo per auto, moto e deposito.",
        "Sala Da Pranzo": "L'ambiente perfetto per condividere pranzi e cene.",
        "Ufficio": "Un angolo riservato per lavorare e concentrarsi da casa.",
        "Corridoio": "Collegamenti ordinati e armoniosi tra gli ambienti della casa.",
    }
    return f"{category}: {defaults.get(category, 'Un ambiente da scoprire e vivere ogni giorno.')}"


available = widgets.SelectMultiple(
    options=list(item_by_key.keys()),
    description="Foto:",
    rows=min(12, max(5, len(item_by_key))),
    layout=widgets.Layout(width="430px"),
)
ordered = widgets.Select(
    options=[],
    description="Ordine:",
    rows=min(12, max(5, len(item_by_key))),
    layout=widgets.Layout(width="430px"),
)

add_button = widgets.Button(description="Aggiungi", button_style="success")
remove_button = widgets.Button(description="Rimuovi")
up_button = widgets.Button(description="Su")
down_button = widgets.Button(description="Giu")
status = widgets.HTML(value="Seleziona le foto e premi Aggiungi.")


def refresh_order(options):
    current = ordered.value
    ordered.options = options
    if current in options:
        ordered.value = current


def add_selected(_):
    current = list(ordered.options)
    for key in available.value:
        if key not in current:
            current.append(key)
    refresh_order(current)
    status.value = f"Foto selezionate: {len(current)}"


def remove_selected(_):
    current = [key for key in ordered.options if key != ordered.value]
    refresh_order(current)
    status.value = f"Foto selezionate: {len(current)}"


def move_selected(delta):
    current = list(ordered.options)
    if ordered.value not in current:
        return
    old_index = current.index(ordered.value)
    new_index = max(0, min(len(current) - 1, old_index + delta))
    current.insert(new_index, current.pop(old_index))
    refresh_order(current)
    ordered.value = current[new_index]


add_button.on_click(add_selected)
remove_button.on_click(remove_selected)
up_button.on_click(lambda _: move_selected(-1))
down_button.on_click(lambda _: move_selected(1))

title = widgets.Text(value="Scopri questa proprieta", description="Titolo:", layout=widgets.Layout(width="520px"))
subtitle = widgets.Text(value="Una casa pensata per il tuo prossimo capitolo", description="Sottotitolo:", layout=widgets.Layout(width="520px"))
price = widgets.Text(value="", description="Prezzo:", layout=widgets.Layout(width="300px"))
cta = widgets.Text(value="Contattaci per una visita", description="CTA:", layout=widgets.Layout(width="520px"))
slide_inputs = {}
slide_editor = widgets.VBox([])
seconds = widgets.FloatSlider(value=3.0, min=1.5, max=8.0, step=0.5, description="Sec/foto:")


def sync_texts(change=None):
    keys = list(ordered.options)
    rows = []
    for index, key in enumerate(keys):
        if key not in slide_inputs:
            slide_inputs[key] = widgets.Text(
                value=default_slide_text(item_by_key[key]),
                layout=widgets.Layout(width="650px"),
            )
        rows.append(widgets.HBox([
            widgets.Label(f"{index + 1:02d}", layout=widgets.Layout(width="35px")),
            widgets.Label(key, layout=widgets.Layout(width="230px")),
            slide_inputs[key],
        ]))
    slide_editor.children = rows


ordered.observe(sync_texts, names="options")

def make_font(size, bold=False):
    font_path = font_manager.findfont(
        font_manager.FontProperties(
            family="DejaVu Sans",
            weight="bold" if bold else "normal"
        )
    )
    return ImageFont.truetype(font_path, size)


def fit_on_vertical_canvas(image_path, width=1080, height=1920):
    image = Image.open(image_path).convert("RGB")
    background = ImageOps.fit(image, (width, height), method=Image.Resampling.LANCZOS)
    background = background.filter(ImageFilter.GaussianBlur(radius=24))
    background = ImageEnhance.Brightness(background).enhance(0.55)

    foreground = ImageOps.contain(image, (width - 96, height - 380), method=Image.Resampling.LANCZOS)
    x = (width - foreground.width) // 2
    y = (height - foreground.height) // 2
    background.paste(foreground, (x, y))
    return background


def wrapped_text(draw, text, font, max_width, max_lines=2):
    words = str(text).split()
    lines = []
    current = ""
    for word in words:
        candidate = f"{current} {word}".strip()
        if draw.textbbox((0, 0), candidate, font=font)[2] <= max_width:
            current = candidate
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    lines = lines[:max_lines]
    if len(words) and len(lines) == max_lines and lines[-1] != " ".join(words):
        lines[-1] = lines[-1].rstrip(" .") + "..."
    return "\n".join(lines)


def add_text_overlay(canvas, slide_text, slide_index, total, show_global_text):
    draw = ImageDraw.Draw(canvas, "RGBA")
    width, height = canvas.size
    font_title = make_font(60, bold=True)
    font_description = make_font(30, bold=False)
    font_small = make_font(10, bold=False)

    panel_y = 36
    if show_global_text:
        draw.rounded_rectangle((36, panel_y, width - 36, 330), radius=32, fill=(0, 0, 0, 155))
        title_y = panel_y + 36
        description_y = title_y + 72
        draw.text(
            (76, title_y),
            wrapped_text(draw, title.value, font_title, width - 150, 1),
            font=font_title,
            fill="white",
        )
        draw.multiline_text(
            (76, description_y),
            wrapped_text(draw, subtitle.value, font_description, width - 150, 2),
            font=font_description,
            fill=(240, 240, 240),
            spacing=8,
        )

        if price.value.strip():
            draw.rounded_rectangle((54, 250, 440, 316), radius=18, fill=(255, 255, 255, 235))
            draw.text((78, 268), price.value.strip(), font=font_description, fill=(20, 20, 20))

        draw.text((width - 530, 268), cta.value, font=font_description, fill=(235, 235, 235))

    draw.rounded_rectangle((36, height - 330, width - 36, height - 44), radius=32, fill=(0, 0, 0, 175))
    draw.multiline_text(
        (76, height - 292),
        wrapped_text(draw, slide_text, font_description, width - 150, 2),
        font=font_description,
        fill="white",
        spacing=8,
    )
    draw.text((width - 150, height - 92), f"{slide_index}/{total}", font=font_small, fill=(235, 235, 235))
    return canvas


render_state = {"running": False}


def render_reel(_):
    if render_state["running"]:
        return

    keys = list(ordered.options)
    if not keys:
        status.value = "Seleziona almeno una foto prima di generare il Reel."
        return

    render_state["running"] = True
    generate_button.disabled = True
    status.value = "Generazione Reel in corso..."
    texts = [slide_inputs[key].value.strip() or default_slide_text(item_by_key[key]) for key in keys]

    clips = []
    try:
        # Keep one deterministic output and avoid showing stale/duplicate reels.
        for old_file in OUTPUT_DIR.glob("real_estate_reel_9x16*.mp4"):
            old_file.unlink()
        for old_frame in OUTPUT_DIR.glob("reel_frame_*.jpg"):
            old_frame.unlink()

        for index, key in enumerate(keys):
            frame = fit_on_vertical_canvas(item_path(item_by_key[key]))
            slide_number = index + 1
            frame = add_text_overlay(
                frame,
                texts[index],
                slide_number,
                len(keys),
                show_global_text=slide_number == 1 or slide_number == len(keys),
            )
            frame_path = OUTPUT_DIR / f"reel_frame_{index + 1:02d}.jpg"
            frame.save(frame_path, quality=95)
            clip = ImageClip(str(frame_path))
            duration = float(seconds.value)
            if hasattr(clip, "with_duration"):
                clip = clip.with_duration(duration)
            else:
                clip = clip.set_duration(duration)
            clips.append(clip)

        output_path = OUTPUT_DIR / "real_estate_reel_9x16.mp4"
        video = concatenate_videoclips(clips, method="compose")
        video.write_videofile(
            str(output_path),
            fps=30,
            codec="libx264",
            audio=False,
            preset="medium",
            threads=2,
            logger=None,
        )
        video.close()
        status.value = f"Reel generato: <code>{output_path}</code>"
        display(Video(str(output_path), embed=True, width=360))
    finally:
        for clip in clips:
            clip.close()
        render_state["running"] = False
        generate_button.disabled = False


generate_button = widgets.Button(description="Genera Reel", button_style="primary", icon="video-camera")
generate_button.on_click(render_reel)

display(widgets.HTML("<h3>Step 17 - Creazione Reel 9:16</h3>"))
display(widgets.HBox([
    widgets.VBox([available, add_button]),
    widgets.VBox([ordered, widgets.HBox([up_button, down_button, remove_button])]),
]))
display(widgets.VBox([
    title,
    subtitle,
    price,
    cta,
    widgets.HTML("<b>Testi per ogni foto</b> (gia compilati, ma modificabili):"),
    slide_editor,
    seconds,
    generate_button,
    status,
]))



# Step 18 - Creazione Brochure

In [ ]:
import json
from io import BytesIO
from pathlib import Path

import ipywidgets as widgets
from IPython.display import FileLink, display
from PIL import Image
from reportlab.lib.colors import HexColor
from reportlab.lib.pagesizes import A4
from reportlab.lib.utils import ImageReader
from reportlab.pdfbase.pdfmetrics import stringWidth
from reportlab.pdfgen import canvas

base_dir = Path(globals().get("BASE_DIR", "/content/real_estate_ai_assistant"))


def find_json(names):
    for name in names:
        matches = list(base_dir.rglob(name))
        if matches:
            return matches[0]
    return None


def load_json(path, default):
    if not path:
        return default
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def as_text(value):
    return "" if value is None else str(value).strip()


def first_text(data, keys):
    if not isinstance(data, dict):
        return ""
    for key in keys:
        value = data.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    for value in data.values():
        if isinstance(value, dict):
            found = first_text(value, keys)
            if found:
                return found
    return ""


# Reuse the data already generated by the previous notebook steps.
images_json = find_json([
    "corrected_images.json",
    "classified_images.json",
    "selected_images.json",
])
images = load_json(
    images_json,
    globals().get("corrected_images")
    or globals().get("classified_images")
    or globals().get("selected_images")
    or [],
)
if not images:
    raise FileNotFoundError("Esegui prima lo step di selezione/correzione immagini.")

content_json = find_json(["generated_content.json"])
generated_content = load_json(content_json, globals().get("generated_content", {}))
listing = (
    globals().get("structured_listing")
    or globals().get("listing_data")
    or globals().get("property_data")
    or {}
)
agency = globals().get("agency_profile", {})
output_dir = base_dir / "output" / "brochures"
output_dir.mkdir(parents=True, exist_ok=True)


def resolve_path(value):
    path = Path(value or "")
    return path if path.is_absolute() else base_dir / path


images = [item for item in images if resolve_path(item.get("file_path")).exists()]
if not images:
    raise FileNotFoundError("Le immagini selezionate non sono piu disponibili nel runtime.")


def listing_value(*keys):
    for key in keys:
        if as_text(listing.get(key)):
            return as_text(listing[key])
    return ""


def listing_bullets(*keys):
    for key in keys:
        value = listing.get(key)
        if isinstance(value, list) and value:
            return "\n".join(f"- {as_text(item)}" for item in value if as_text(item))
        if as_text(value):
            return as_text(value)
    return ""


def make_preview(path):
    image = Image.open(path).convert("RGB")
    image.thumbnail((180, 125), Image.Resampling.LANCZOS)
    buffer = BytesIO()
    image.save(buffer, format="JPEG", quality=88)
    return widgets.Image(value=buffer.getvalue(), format="jpeg", width=180, height=125)


brand_colors = agency.get("brand_colors", [])
primary_color = brand_colors[0] if brand_colors else "#163B5C"
secondary_color = brand_colors[1] if len(brand_colors) > 1 else "#C89B3C"
agency_name = as_text(agency.get("agency_name") or agency.get("name")) or "La tua agenzia"

# These values are generated by AI in the content step, then remain editable.
ai_title = first_text(generated_content, ["brochure_title", "property_title", "headline", "title", "titolo"])
ai_intro = first_text(generated_content, ["brochure_description", "property_description", "long_description", "description", "summary", "descrizione_breve"])
ai_highlights = first_text(generated_content, ["brochure_highlights", "highlights", "key_features", "selling_points"])
ai_closing = first_text(generated_content, ["brochure_closing", "closing_text", "cta", "call_to_action"])

title = widgets.Text(
    value=ai_title or listing_value("titolo", "title", "property_title", "headline") or "Una proprieta da scoprire",
    description="Titolo AI:",
    layout=widgets.Layout(width="800px"),
)
location = widgets.Text(
    value=listing_value("localita", "location", "address", "city"),
    description="Localita:",
    layout=widgets.Layout(width="800px"),
)
price = widgets.Text(
    value=listing_value("prezzo", "price", "formatted_price"),
    description="Prezzo:",
    layout=widgets.Layout(width="420px"),
)
intro = widgets.Textarea(
    value=ai_intro or listing_value("descrizione_breve", "description", "summary", "short_description"),
    description="Descrizione AI:",
    layout=widgets.Layout(width="800px", height="180px"),
)
highlights = widgets.Textarea(
    value=ai_highlights or listing_bullets("punti_forza"),
    description="Punti forza AI:",
    layout=widgets.Layout(width="800px", height="135px"),
)
closing = widgets.Textarea(
    value=ai_closing or "Contattaci per ricevere maggiori informazioni o fissare una visita.",
    description="Testo finale:",
    layout=widgets.Layout(width="800px", height="100px"),
)
contact = widgets.Text(
    value=as_text(agency.get("phone") or agency.get("email")) or "Contattaci per maggiori informazioni",
    description="Contatto:",
    layout=widgets.Layout(width="800px"),
)

selection_widgets = []
cards = []
for index, item in enumerate(images, start=1):
    checkbox = widgets.Checkbox(value=True, description=f"Immagine {index}", indent=False)
    selection_widgets.append({"item": item, "checkbox": checkbox})
    cards.append(widgets.VBox([
        make_preview(resolve_path(item.get("file_path"))),
        checkbox,
    ]))

selection_grid = widgets.GridBox(
    cards,
    layout=widgets.Layout(grid_template_columns="repeat(4, 205px)", grid_gap="16px", padding="8px"),
)
confirm_images_button = widgets.Button(description="Conferma immagini", button_style="primary")
generate_button = widgets.Button(description="Genera brochure PDF", button_style="success", disabled=True)
result_output = widgets.Output()
selected_images = []
render_state = {"running": False}


def prepare_brochure(_):
    global selected_images
    selected_images = [entry["item"] for entry in selection_widgets if entry["checkbox"].value]
    if not selected_images:
        with result_output:
            result_output.clear_output()
            print("Seleziona almeno un'immagine.")
        return
    generate_button.disabled = False
    with result_output:
        result_output.clear_output()
        print(f"{len(selected_images)} immagini selezionate. Controlla i testi AI e poi genera il PDF.")


def wrap_text(text, font_name, font_size, max_width):
    words = as_text(text).split()
    lines, current = [], ""
    for word in words:
        candidate = f"{current} {word}".strip()
        if stringWidth(candidate, font_name, font_size) <= max_width:
            current = candidate
        else:
            if current:
                lines.append(current)
            current = word
    if current:
        lines.append(current)
    return lines or [""]


def draw_wrapped(pdf, text, x, y, max_width, font_name, font_size, leading, color):
    pdf.setFillColor(color)
    pdf.setFont(font_name, font_size)
    for line in wrap_text(text, font_name, font_size, max_width):
        pdf.drawString(x, y, line)
        y -= leading
    return y


def draw_footer(pdf, page_width, page_number):
    pdf.setFillColor(HexColor(primary_color))
    pdf.rect(0, 0, page_width, 38, stroke=0, fill=1)
    pdf.setFillColor("white")
    pdf.setFont("Helvetica", 9)
    pdf.drawString(34, 14, agency_name)
    pdf.drawCentredString(page_width / 2, 14, contact.value.strip())
    pdf.drawRightString(page_width - 34, 14, f"Pagina {page_number}")


def draw_cover(pdf, page_width, page_height, logo_path):
    pdf.setFillColor(HexColor("#F7F6F2"))
    pdf.rect(0, 0, page_width, page_height, stroke=0, fill=1)
    if logo_path.exists():
        try:
            logo = Image.open(logo_path).convert("RGBA")
            logo.thumbnail((220, 105), Image.Resampling.LANCZOS)
            buffer = BytesIO()
            logo.save(buffer, format="PNG")
            pdf.drawImage(ImageReader(buffer), 44, page_height - 145, logo.width, logo.height, mask="auto")
        except Exception:
            pass

    hero = Image.open(resolve_path(selected_images[0].get("file_path"))).convert("RGB")
    hero.thumbnail((page_width - 88, 410), Image.Resampling.LANCZOS)
    hero_buffer = BytesIO()
    hero.save(hero_buffer, format="JPEG", quality=94)
    hero_x = (page_width - hero.width) / 2
    hero_y = 122
    pdf.drawImage(ImageReader(hero_buffer), hero_x, hero_y, hero.width, hero.height)
    panel_width = min(235, hero.width * 0.52)
    pdf.setFillColor(HexColor(primary_color))
    pdf.rect(hero_x, hero_y, panel_width, hero.height, stroke=0, fill=1)
    pdf.setFillColor("white")
    draw_wrapped(pdf, location.value or "LOCATION", hero_x + 20, hero_y + hero.height - 58, panel_width - 38, "Helvetica", 12, 16, "white")
    draw_wrapped(pdf, title.value, hero_x + 20, hero_y + hero.height - 105, panel_width - 38, "Helvetica-Bold", 22, 28, "white")
    pdf.setStrokeColor("white")
    pdf.setLineWidth(1.5)
    pdf.line(hero_x + 20, hero_y + 140, hero_x + panel_width - 20, hero_y + 140)
    pdf.setFont("Helvetica", 12)
    pdf.drawString(hero_x + 20, hero_y + 105, "PREZZO")
    pdf.setFont("Helvetica-Bold", 19)
    pdf.drawString(hero_x + 20, hero_y + 75, price.value.strip() or "Su richiesta")
    draw_footer(pdf, page_width, 1)
    pdf.showPage()


def draw_description_page(pdf, page_width, page_height, page_number):
    pdf.setFillColor(HexColor(primary_color))
    pdf.setFont("Helvetica-Bold", 26)
    pdf.drawCentredString(page_width / 2, page_height - 70, "DESCRIZIONE")
    y = draw_wrapped(pdf, intro.value, 56, page_height - 125, page_width - 112, "Helvetica", 13, 21, HexColor("#202020"))
    if highlights.value.strip():
        y -= 16
        pdf.setFillColor(HexColor(primary_color))
        pdf.setFont("Helvetica-Bold", 16)
        pdf.drawString(56, y, "PUNTI DI FORZA")
        draw_wrapped(pdf, highlights.value, 56, y - 30, page_width - 112, "Helvetica", 12, 19, HexColor("#202020"))
    draw_footer(pdf, page_width, page_number)
    pdf.showPage()


def crop_to_box(image_path, box_width, box_height):
    image = Image.open(image_path).convert("RGB")
    source_ratio = image.width / image.height
    box_ratio = box_width / box_height
    if source_ratio > box_ratio:
        new_height = image.height
        new_width = int(new_height * box_ratio)
        left = (image.width - new_width) // 2
        image = image.crop((left, 0, left + new_width, new_height))
    else:
        new_width = image.width
        new_height = int(new_width / box_ratio)
        top = (image.height - new_height) // 2
        image = image.crop((0, top, new_width, top + new_height))
    image = image.resize((int(box_width), int(box_height)), Image.Resampling.LANCZOS)
    buffer = BytesIO()
    image.save(buffer, format="JPEG", quality=94)
    return buffer


def draw_collage_page(pdf, page_width, page_height, group, page_number):
    margin, gap = 42, 10
    usable_width = page_width - (margin * 2)
    usable_height = page_height - 128
    slots = []
    if len(group) == 1:
        slots = [(margin, 76, usable_width, usable_height)]
    elif len(group) == 2:
        half = (usable_height - gap) / 2
        slots = [(margin, 76 + half + gap, usable_width, half), (margin, 76, usable_width, half)]
    elif len(group) == 3:
        top_height = usable_height * 0.56
        bottom_height = usable_height - top_height - gap
        half_width = (usable_width - gap) / 2
        slots = [
            (margin, 76 + bottom_height + gap, usable_width, top_height),
            (margin, 76, half_width, bottom_height),
            (margin + half_width + gap, 76, half_width, bottom_height),
        ]
    else:
        half_width = (usable_width - gap) / 2
        half_height = (usable_height - gap) / 2
        slots = [
            (margin, 76 + half_height + gap, half_width, half_height),
            (margin + half_width + gap, 76 + half_height + gap, half_width, half_height),
            (margin, 76, half_width, half_height),
            (margin + half_width + gap, 76, half_width, half_height),
        ]

    for item, (x, y, width, height) in zip(group, slots):
        image_buffer = crop_to_box(resolve_path(item.get("file_path")), width, height)
        pdf.drawImage(ImageReader(image_buffer), x, y, width, height)
    draw_footer(pdf, page_width, page_number)
    pdf.showPage()


def draw_closing_page(pdf, page_width, page_height, page_number, logo_path):
    pdf.setFillColor(HexColor("#F7F6F2"))
    pdf.rect(0, 0, page_width, page_height, stroke=0, fill=1)
    pdf.setFillColor(HexColor(primary_color))
    pdf.setFont("Helvetica-Bold", 25)
    pdf.drawCentredString(page_width / 2, page_height - 105, "TI PIACE QUESTA PROPRIETA?")
    pdf.setFont("Helvetica-Bold", 18)
    pdf.drawCentredString(page_width / 2, page_height - 140, "Contattaci per non perdere l'occasione.")
    draw_wrapped(pdf, closing.value, 64, page_height - 210, page_width - 128, "Helvetica", 14, 22, HexColor("#202020"))
    if logo_path.exists():
        try:
            logo = Image.open(logo_path).convert("RGBA")
            logo.thumbnail((170, 90), Image.Resampling.LANCZOS)
            buffer = BytesIO()
            logo.save(buffer, format="PNG")
            pdf.drawImage(ImageReader(buffer), page_width - logo.width - 54, 105, logo.width, logo.height, mask="auto")
        except Exception:
            pass
    draw_footer(pdf, page_width, page_number)
    pdf.showPage()


def generate_brochure(_):
    if render_state["running"] or not selected_images:
        return
    render_state["running"] = True
    generate_button.disabled = True
    try:
        output_path = output_dir / "brochure_immobile.pdf"
        pdf = canvas.Canvas(str(output_path), pagesize=A4)
        page_width, page_height = A4
        logo_path = resolve_path(agency.get("logo_path"))
        draw_cover(pdf, page_width, page_height, logo_path)
        draw_description_page(pdf, page_width, page_height, 2)
        page_number = 3
        for start in range(0, len(selected_images), 4):
            draw_collage_page(pdf, page_width, page_height, selected_images[start:start + 4], page_number)
            page_number += 1
        draw_closing_page(pdf, page_width, page_height, page_number, logo_path)
        pdf.save()
        with result_output:
            result_output.clear_output()
            print("Brochure PDF creata correttamente:")
            display(FileLink(str(output_path)))
    finally:
        render_state["running"] = False
        generate_button.disabled = False


confirm_images_button.on_click(prepare_brochure)
generate_button.on_click(generate_brochure)

display(widgets.VBox([
    widgets.HTML("<h3>Step 18 - Creazione Brochure PDF</h3>"),
    widgets.HTML("<b>1.</b> Seleziona le immagini da usare nei collage."),
    selection_grid,
    confirm_images_button,
    widgets.HTML("<b>2.</b> I testi arrivano dallo step AI e sono modificabili prima della generazione."),
    title,
    location,
    price,
    intro,
    highlights,
    closing,
    contact,
    generate_button,
    result_output,
]))


# Step 19 - Caption Social Media

In [ ]:
import json
import time
from pathlib import Path

import ipywidgets as widgets
import requests
from IPython.display import display

base_dir = Path(globals().get("BASE_DIR", "/content/real_estate_ai_assistant"))


def find_json(names):
    for name in names:
        matches = list(base_dir.rglob(name))
        if matches:
            return matches[0]
    return None


def load_json(path, default):
    if not path:
        return default
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def as_text(value):
    return "" if value is None else str(value).strip()


def first_text(data, keys):
    if not isinstance(data, dict):
        return ""
    for key in keys:
        value = data.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    for value in data.values():
        if isinstance(value, dict):
            found = first_text(value, keys)
            if found:
                return found
    return ""


output_dir = base_dir / "output" / "social"
output_dir.mkdir(parents=True, exist_ok=True)

content_json = find_json(["generated_content.json"])
generated_content = load_json(content_json, globals().get("generated_content", {}))
listing = (
    globals().get("structured_listing")
    or globals().get("listing_data")
    or globals().get("property_data")
    or {}
)
agency = globals().get("agency_profile", {})


def listing_value(*keys):
    for key in keys:
        if as_text(listing.get(key)):
            return as_text(listing[key])
    return ""


property_title = (
    first_text(generated_content, ["brochure_title", "property_title", "headline", "title", "titolo"])
    or listing_value("titolo", "title", "property_title", "headline")
    or "Questa proprieta"
)
property_description = (
    first_text(generated_content, ["brochure_description", "property_description", "description", "summary", "descrizione_breve"])
    or listing_value("descrizione_breve", "description", "summary", "short_description")
)
location = listing_value("localita", "location", "city", "address")
price_value = listing_value("prezzo", "price", "formatted_price")
agency_name = as_text(agency.get("agency_name") or agency.get("name")) or "la nostra agenzia"

try:
    from google.colab import userdata
    secret_key = userdata.get("Real_Estate_AI_Assistant")
except Exception:
    secret_key = None

api_key = (
    globals().get("OPENROUTER_API_KEY")
    or globals().get("openrouter_api_key")
    or secret_key
)
model_name = (
    globals().get("MODEL_NAME")
    or globals().get("PRIMARY_MODEL")
    or "google/gemma-4-31b-it:free"
)
fallback_models = [
    model_name,
    "nvidia/nemotron-3-super-120b-a12b:free",
    "qwen/qwen3-next-80b-a3b-instruct:free",
]

instagram_checkbox = widgets.Checkbox(value=True, description="Instagram")
facebook_checkbox = widgets.Checkbox(value=True, description="Facebook")
linkedin_checkbox = widgets.Checkbox(value=True, description="LinkedIn")
tone = widgets.Dropdown(
    options=["Professionale e coinvolgente", "Elegante e premium", "Caldo e familiare", "Diretto e commerciale"],
    value="Professionale e coinvolgente",
    description="Tono:",
    layout=widgets.Layout(width="420px"),
)
cta = widgets.Text(
    value="Contattaci per maggiori informazioni o per fissare una visita.",
    description="CTA:",
    layout=widgets.Layout(width="820px"),
)
generate_button = widgets.Button(description="Genera caption AI", button_style="primary", icon="magic")
save_button = widgets.Button(description="Salva versioni finali", button_style="success", disabled=True)
status = widgets.Output()
editor = widgets.VBox()
caption_inputs = {}
original_content = {}


def selected_platforms():
    selected = []
    if instagram_checkbox.value:
        selected.append("instagram")
    if facebook_checkbox.value:
        selected.append("facebook")
    if linkedin_checkbox.value:
        selected.append("linkedin")
    return selected


def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1]
        text = text.rsplit("```", 1)[0]
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("La risposta AI non contiene un JSON valido.")
    return json.loads(text[start:end + 1])


def fallback_caption(platform):
    hashtag_base = "#immobiliare #casainvendita #realestate #agenziaimmobiliare"
    location_tag = f" #{location.replace(' ', '')}" if location else ""
    return {
        "caption": f"{property_title}. {property_description} {cta.value.strip()}",
        "hashtags": hashtag_base + location_tag,
        "platform": platform,
    }


def show_editors(content_by_platform):
    global caption_inputs, original_content
    original_content = content_by_platform
    caption_inputs = {}
    elements = [widgets.HTML("<h3>Modifica caption e hashtag prima del salvataggio</h3>")]
    for platform in selected_platforms():
        item = content_by_platform.get(platform, fallback_caption(platform))
        caption_widget = widgets.Textarea(
            value=as_text(item.get("caption")),
            description="Caption:",
            layout=widgets.Layout(width="820px", height="170px"),
        )
        hashtag_widget = widgets.Textarea(
            value=as_text(item.get("hashtags")),
            description="Hashtag:",
            layout=widgets.Layout(width="820px", height="95px"),
        )
        caption_inputs[platform] = {"caption": caption_widget, "hashtags": hashtag_widget}
        elements.extend([
            widgets.HTML(f"<h4>{platform.title()}</h4>"),
            caption_widget,
            hashtag_widget,
        ])
    editor.children = elements
    save_button.disabled = False


def generate_captions(_):
    platforms = selected_platforms()
    if not platforms:
        with status:
            status.clear_output()
            print("Seleziona almeno una piattaforma.")
        return

    generate_button.disabled = True
    try:
        if not api_key:
            raise ValueError("Non trovo la chiave OpenRouter. Esegui prima lo step di configurazione API.")

        prompt = f"""
Sei un copywriter immobiliare italiano per {agency_name}.
Genera contenuti social per questo immobile.

Titolo: {property_title}
Localita: {location or 'non indicata'}
Prezzo: {price_value or 'non indicato'}
Descrizione: {property_description or 'non disponibile'}
Call to action: {cta.value.strip()}
Tono: {tone.value}
Piattaforme: {', '.join(platforms)}

Rispondi esclusivamente con JSON valido, senza markdown, nel formato:
{{
  "instagram": {{"caption": "...", "hashtags": "#..."}},
  "facebook": {{"caption": "...", "hashtags": "#..."}},
  "linkedin": {{"caption": "...", "hashtags": "#..."}}
}}

Regole:
- Inserisci solo le piattaforme richieste.
- Scrivi in italiano.
- Instagram: caption max 900 caratteri e 12-18 hashtag pertinenti.
- Facebook: caption max 700 caratteri e 3-6 hashtag.
- LinkedIn: caption max 900 caratteri, professionale, e 3-5 hashtag.
- Non inventare dettagli dell'immobile.
"""
        result = None
        last_error = None
        for attempt, current_model in enumerate(dict.fromkeys(fallback_models), start=1):
            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
                json={
                    "model": current_model,
                    "messages": [{"role": "user", "content": prompt}],
                    "temperature": 0.7,
                },
                timeout=90,
            )
            if response.status_code != 429:
                response.raise_for_status()
                result = response.json()
                break

            last_error = response
            # Free models can be briefly rate-limited. Wait before the next fallback.
            wait_seconds = int(response.headers.get("Retry-After", 5 * attempt))
            time.sleep(min(wait_seconds, 20))

        if result is None:
            raise RuntimeError(
                "OpenRouter sta limitando temporaneamente tutti i modelli gratuiti. "
                "Attendi un minuto e riprova."
            ) from last_error

        raw_content = result["choices"][0]["message"]["content"]
        show_editors(extract_json(raw_content))
        with status:
            status.clear_output()
            print("Caption generate. Puoi modificarle prima di salvarle.")
    except Exception as error:
        with status:
            status.clear_output()
            print("Errore nella generazione:", error)
    finally:
        generate_button.disabled = False


def save_captions(_):
    if not caption_inputs:
        return
    edited_content = {
        platform: {
            "caption": fields["caption"].value.strip(),
            "hashtags": fields["hashtags"].value.strip(),
        }
        for platform, fields in caption_inputs.items()
    }
    output_path = output_dir / "social_caption_versions.json"
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(
            {
                "property_title": property_title,
                "original_ai_content": original_content,
                "edited_content": edited_content,
            },
            file,
            indent=4,
            ensure_ascii=False,
        )
    with status:
        status.clear_output()
        print("Caption e hashtag salvati in:", output_path)


generate_button.on_click(generate_captions)
save_button.on_click(save_captions)

display(widgets.VBox([
    widgets.HTML("<h3>Step 19 - Caption e hashtag social</h3>"),
    widgets.HTML(f"<b>Immobile:</b> {property_title}"),
    widgets.HBox([instagram_checkbox, facebook_checkbox, linkedin_checkbox]),
    tone,
    cta,
    generate_button,
    status,
    editor,
    save_button,
]))


# Step 20 - Chiusura e download file

In [ ]:
import json
import zipfile
from pathlib import Path

import ipywidgets as widgets
from IPython.display import FileLink, display


base_dir = Path(globals().get("BASE_DIR", "/content"))
output_root = base_dir / "output"
export_dir = output_root / "exports"
export_dir.mkdir(parents=True, exist_ok=True)

asset_groups = {
    "Post statico": {
        "folder": output_root / "posts",
        "extensions": {".jpg", ".jpeg", ".png"},
    },
    "Carosello": {
        "folder": output_root / "carousels",
        "extensions": {".jpg", ".jpeg", ".png", ".json"},
    },
    "Reel": {
        "folders": [output_root / "reels", output_root / "reel"],
        "extensions": {".mp4"},
    },
    "Brochure": {
        "folder": output_root / "brochures",
        "extensions": {".pdf"},
    },
    "Caption e hashtag": {
        "folder": output_root / "social",
        "extensions": {".json", ".txt"},
    },
}


def discover_assets():
    assets = {}
    for group_name, config in asset_groups.items():
        folders = config.get("folders", [config.get("folder")])
        assets[group_name] = sorted(
            path
            for folder in folders
            if folder and folder.exists()
            for path in folder.rglob("*")
            if path.is_file() and path.suffix.lower() in config["extensions"]
        )
    return assets


assets = discover_assets()
status = widgets.Output()
create_zip_button = widgets.Button(
    description="Crea ZIP finale",
    button_style="success",
    icon="download",
)


def show_assets(current_assets):
    for group_name, files in current_assets.items():
        print(f"\n{group_name}: {len(files)} file")
        for path in files:
            display(FileLink(str(path)))


def create_export(_):
    current_assets = discover_assets()
    files_to_export = [
        (group_name, path)
        for group_name, files in current_assets.items()
        for path in files
    ]
    if not files_to_export:
        with status:
            status.clear_output()
            print("Non trovo ancora asset finali. Esegui prima gli Step 14-19.")
        return

    create_zip_button.disabled = True
    try:
        zip_path = export_dir / "materiali_marketing_immobile.zip"
        manifest_path = export_dir / "manifest_materiali.json"
        manifest = {group_name: [str(path) for path in files] for group_name, files in current_assets.items()}
        with manifest_path.open("w", encoding="utf-8") as file:
            json.dump(manifest, file, indent=4, ensure_ascii=False)

        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
            for group_name, path in files_to_export:
                archive.write(path, arcname=f"{group_name}/{path.name}")
            archive.write(manifest_path, arcname="manifest_materiali.json")

        with status:
            status.clear_output()
            print("Pacchetto finale creato correttamente:")
            display(FileLink(str(zip_path)))
            print("Manifest dei file inclusi:")
            display(FileLink(str(manifest_path)))
    finally:
        create_zip_button.disabled = False


create_zip_button.on_click(create_export)

with status:
    print("Asset individuati:")
    show_assets(assets)

display(widgets.VBox([
    widgets.HTML("<h3>Step 20 - Riepilogo e download materiali</h3>"),
    widgets.HTML("Controlla gli asset prodotti e crea un unico file ZIP da condividere o archiviare."),
    create_zip_button,
    status,
]))
